First lines to setup notebook.

In [1]:
!pip3 install torch torchvision --index-url https://download.pytorch.org/whl/cu126

Looking in indexes: https://download.pytorch.org/whl/cu126


In [1]:
import torch
from torch import nn

In [13]:
class CustomResidualBlock(nn.Module):
  def __init__(self, in_channels, out_channels, kernel_size=3):
    super().__init__()

    self.shortcut = nn.Identity()

    if in_channels != out_channels:
      self.shortcut = nn.Conv2d(in_channels, out_channels, 1)

    self.first = nn.Conv2d(in_channels, out_channels=out_channels, kernel_size=kernel_size, stride=1, padding=1)
    self.second = nn.Conv2d(out_channels, out_channels=out_channels, kernel_size=kernel_size, stride=1, padding=1)
    self.bn = customBatchNormalization()
    self.relu = nn.ReLU()

  def forward(self, x):
    out = self.bn(self.second(self.bn(self.first(x))))
    return self.relu(out + self.shortcut(x))

In [17]:
class customBatchNormalization(nn.Module):
  def __init__(self, epsilon = 0.000001):
    super().__init__()
    self.epsilon = epsilon
    self.gamma = nn.Parameter(torch.tensor([1.0]))
    self.beta = nn.Parameter(torch.tensor([0.0]))


  def forward(self, x):
    # mean over (0, 2, 3) gives us average across all values for each channel c (B, C, W, H)
    mean = x.mean(dim=(0, 2, 3), keepdim=True)
    var = x.var(dim=(0, 2, 3), keepdim=True)
    normalized = (x - mean) / (self.epsilon + var**(1/2))
    return self.gamma * normalized + self.beta



In [4]:
inp = torch.randint(low=1, high=10, size=(32, 3, 224, 224)).to(torch.float32)

In [5]:
inp.shape

torch.Size([32, 3, 224, 224])

In [19]:
cb = customBatchNormalization()

In [20]:
class ResNet18(nn.Module):
  def __init__(self):
    super().__init__()

    self.initial_convolution = nn.Conv2d(in_channels=3, out_channels=64, kernel_size=7, stride=2)
    self.initial_maxpool = nn.modules.pooling.MaxPool2d(kernel_size=3, stride=2)
    self.first_crb = nn.Sequential(CustomResidualBlock(64, 64), *[CustomResidualBlock(64, 64) for _ in range(2)])
    self.second_crb = nn.Sequential(CustomResidualBlock(64, 128), *[CustomResidualBlock(128, 128) for _ in range(2)])
    self.third_crb = nn.Sequential(CustomResidualBlock(128, 256), *[CustomResidualBlock(256, 256) for _ in range(2)])
    self.fourth_crb = nn.Sequential(CustomResidualBlock(256, 512), *[CustomResidualBlock(512, 512) for _ in range(2)])
    self.avg_pool = nn.AvgPool2d(2)
    self.linear = nn.Flatten()
    self.fc1 = nn.Linear(373248, 1000)
    self.bn = customBatchNormalization()

    self.resnet = nn.Sequential(
        self.bn,
        self.initial_convolution,
        self.initial_maxpool,
        self.first_crb,
        self.second_crb,
        self.third_crb,
        self.fourth_crb,
        self.avg_pool,
        self.linear,
        self.fc1,
    )
  def forward(self, x):
    return self.resnet(x)


In [21]:
rn = ResNet18()

In [22]:
a = rn(inp)

In [23]:
a.shape

torch.Size([32, 1000])